# LeetCode 230 — Kth Smallest Element in a BST bằng Python

Link đề: https://leetcode.com/problems/kth-smallest-element-in-a-bst/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def kthSmallest(self, root: Optional[TreeNode], k: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải độc lập (sắp xếp tập giá trị đã dùng để dựng cây, không đọc cây).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho gốc của một **cây tìm kiếm nhị phân (BST)** và số nguyên `k`; trả về giá trị **nhỏ thứ `k`** (đếm từ `1`) trong tất cả giá trị của cây.
2. Biết **tự dựng cây** trong Python (`TreeNode`) từ danh sách level-order có `None` (`build_tree`), duyệt cây bằng đệ quy và bằng ngăn xếp.
3. Nhận ra **tính chất then chốt của BST**: duyệt **inorder** (trái → gốc → phải) cho các giá trị **theo thứ tự tăng dần**, nên phần tử nhỏ thứ `k` là phần tử thứ `k` được thăm.
4. Đi từ **thu mọi giá trị rồi sắp xếp** → **inorder đệ quy lấy cả danh sách** → **inorder bằng ngăn xếp dừng sớm** (cách nộp bài) → **đánh dấu kích thước cây con (follow up: truy vấn `O(h)`)** → **Morris traversal (`O(1)` bộ nhớ phụ)**.
5. Tránh các lỗi kinh điển: dùng preorder thay vì inorder, lệch một đơn vị (`k` đếm từ 1), không dừng sớm / không đi sang phải sau khi trừ `k`, và **đệ quy quá sâu** trên cây lệch.

## 2. Đề bài & ví dụ

Cho `root` của một cây tìm kiếm nhị phân và một số nguyên `k`. Trả về **giá trị nhỏ thứ `k`** (tính từ `1`) trong tất cả các giá trị của các nút.

(BST: với mỗi nút, mọi giá trị ở cây con **trái** nhỏ hơn giá trị của nút, mọi giá trị ở cây con **phải** lớn hơn.)

**Ví dụ 1**

- Input: `root = [3,1,4,null,2]`, `k = 1`
- Output: `1`
- Giải thích: các giá trị `1, 2, 3, 4`; nhỏ nhất là `1`.

**Ví dụ 2**

- Input: `root = [5,3,6,2,4,null,null,1]`, `k = 3`
- Output: `3`
- Giải thích: các giá trị sắp tăng `1, 2, 3, 4, 5, 6`; nhỏ thứ ba là `3`.

**Ràng buộc:** số nút `n`; `1 <= k <= n <= 10⁴`; `0 <= Node.val <= 10⁴`.

**Follow up:** nếu BST bị sửa thường xuyên (chèn, xóa) và phải tìm phần tử nhỏ thứ `k` thường xuyên, làm sao tối ưu?

(Dạng `[3,1,4,null,2]` là danh sách **level-order** (theo từng tầng, từ trái sang phải), `null` là nút vắng mặt.)

In [1]:
from typing import List, Optional
import random
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ danh sách level-order (None = nút vắng mặt), giống định dạng của LeetCode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> danh sách level-order (cắt các None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([3, 1, 4, None, 2], 1, 1),
    ([5, 3, 6, 2, 4, None, None, 1], 3, 3),
]
for values, k, expected in examples:
    assert tree_to_list(build_tree(values)) == values
    print(f"root={values}, k={k}, kỳ vọng {expected}")

root=[3, 1, 4, None, 2], k=1, kỳ vọng 1
root=[5, 3, 6, 2, 4, None, None, 1], k=3, kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Nút cây `TreeNode` và cách dựng cây

Mỗi nút có `val` (giá trị), `left` (con trái) và `right` (con phải); con vắng mặt là `None`. Có thể dựng cây "bằng tay" bằng cách nối các nút. LeetCode cho cây dưới dạng danh sách level-order; hàm `build_tree` ở trên dùng một **hàng đợi** (`deque`) để gán con trái/phải cho từng nút theo thứ tự tầng.

In [2]:
# Dựng tay cây [3,1,4,null,2]
root = TreeNode(3)
root.left = TreeNode(1)
root.right = TreeNode(4)
root.left.right = TreeNode(2)
print(tree_to_list(root))                      # [3, 1, 4, None, 2]
print(tree_to_list(build_tree([3, 1, 4, None, 2])) == tree_to_list(root))   # True
print(root.val, root.left.val, root.left.right.val, root.right.left)         # 3 1 2 None

[3, 1, 4, None, 2]
True
3 1 2 None


### 3.2. Cây tìm kiếm nhị phân (BST) và tính chất sắp thứ tự

Trong BST, mọi giá trị ở cây con trái **nhỏ hơn** giá trị nút và mọi giá trị ở cây con phải **lớn hơn**. Hàm `bst_insert` dưới đây chèn một giá trị bằng cách đi xuống: nhỏ hơn thì sang trái, lớn hơn thì sang phải, gặp chỗ trống thì đặt vào (ta sẽ dùng để sinh cây ngẫu nhiên khi test).

In [3]:
def bst_insert(root, val):
    """Chèn val vào BST (lặp, không đệ quy); trả về gốc."""
    if root is None:
        return TreeNode(val)
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = TreeNode(val)
                return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = TreeNode(val)
                return root
            cur = cur.right


root = None
for v in [5, 3, 6, 2, 4, 1]:
    root = bst_insert(root, v)
print(tree_to_list(root))                      # [5, 3, 6, 2, 4, None, None, 1]

[5, 3, 6, 2, 4, None, None, 1]


### 3.3. Duyệt cây theo thứ tự: preorder, inorder, postorder

Duyệt bằng đệ quy ba kiểu, khác nhau ở thời điểm "thăm" nút:

- **preorder:** gốc → trái → phải;
- **inorder:** trái → gốc → phải;
- **postorder:** trái → phải → gốc.

Trên BST, **inorder** liệt kê các giá trị **theo thứ tự tăng dần**.

In [4]:
def preorder(node, out):
    if node:
        out.append(node.val)
        preorder(node.left, out)
        preorder(node.right, out)
    return out


def inorder(node, out):
    if node:
        inorder(node.left, out)
        out.append(node.val)
        inorder(node.right, out)
    return out


def postorder(node, out):
    if node:
        postorder(node.left, out)
        postorder(node.right, out)
        out.append(node.val)
    return out


root = build_tree([5, 3, 6, 2, 4, None, None, 1])
print("preorder :", preorder(root, []))      # [5, 3, 2, 1, 4, 6]
print("inorder  :", inorder(root, []))       # [1, 2, 3, 4, 5, 6]  <- tăng dần
print("postorder:", postorder(root, []))     # [1, 2, 4, 3, 6, 5]

preorder : [5, 3, 2, 1, 4, 6]
inorder  : [1, 2, 3, 4, 5, 6]
postorder: [1, 2, 4, 3, 6, 5]


### 3.4. Inorder bằng ngăn xếp (stack) thay cho đệ quy

Đệ quy dùng "ngăn xếp lời gọi" ngầm. Ta có thể tự quản lý bằng một list `stack`: đi sang trái đến hết (đẩy từng nút vào stack); khi không đi được nữa, `pop` một nút (thăm nó), rồi chuyển sang con phải của nó. Cách này không bị giới hạn độ sâu đệ quy của Python (khoảng 1000).

In [5]:
def inorder_iterative(root):
    out, stack, cur = [], [], root
    while cur or stack:
        if cur:
            stack.append(cur)          # đi sang trái, ghi nhớ đường về
            cur = cur.left
        else:
            cur = stack.pop()          # hết bên trái: thăm nút này
            out.append(cur.val)
            cur = cur.right            # rồi sang cây con phải
    return out


print(inorder_iterative(root))         # [1, 2, 3, 4, 5, 6]

[1, 2, 3, 4, 5, 6]


### 3.5. Giới hạn đệ quy của Python

Mặc định Python cho tối đa khoảng `1000` tầng đệ quy. Một BST **lệch** (chèn các giá trị đã sắp) có độ sâu bằng `n`; với `n` lớn, đệ quy sẽ gây `RecursionError`. Ngăn xếp tự quản lý (mục 3.4) tránh được lỗi này.

In [6]:
import sys
print("giới hạn đệ quy:", sys.getrecursionlimit())

chain = None
for v in range(2000):                  # chèn tăng dần -> cây lệch phải, sâu 2000
    chain = bst_insert(chain, v)
try:
    inorder(chain, [])
except RecursionError:
    print("RecursionError: đệ quy inorder quá sâu trên cây lệch 2000 tầng")
print("inorder bằng ngăn xếp vẫn chạy:", len(inorder_iterative(chain)), "giá trị")

giới hạn đệ quy: 1000
RecursionError: đệ quy inorder quá sâu trên cây lệch 2000 tầng
inorder bằng ngăn xếp vẫn chạy: 2000 giá trị


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây ví dụ 2: `[5,3,6,2,4,null,null,1]`.

```
        5
       / \
      3   6
     / \
    2   4
   /
  1
```

Muốn biết giá trị nhỏ thứ 3, ta liệt kê các giá trị theo thứ tự tăng dần bằng cách duyệt **trái → gốc → phải**:

1. đi sâu nhất sang trái từ `5`: `5 → 3 → 2 → 1`; `1` không còn con trái: thăm `1` (nhỏ thứ **1**);
2. `1` không có con phải, quay về `2`: thăm `2` (nhỏ thứ **2**);
3. `2` không có con phải, quay về `3`: thăm `3` (nhỏ thứ **3**) — đây là đáp án.

Ta không cần duyệt hết cây: dừng ngay khi thăm đủ `k` nút.

In [7]:
root = build_tree([5, 3, 6, 2, 4, None, None, 1])
order = inorder(root, [])
for rank, v in enumerate(order, start=1):
    print(f"nhỏ thứ {rank}: {v}")
k = 3
print("nhỏ thứ", k, "là", order[k - 1])

nhỏ thứ 1: 1
nhỏ thứ 2: 2
nhỏ thứ 3: 3
nhỏ thứ 4: 4
nhỏ thứ 5: 5
nhỏ thứ 6: 6
nhỏ thứ 3 là 3


## 5. Cách 1 — Brute Force / baseline: thu mọi giá trị rồi sắp xếp

**Ý tưởng.** Không dùng tính chất BST: duyệt cây **theo bất kỳ thứ tự nào** (ở đây preorder bằng ngăn xếp) để gom mọi giá trị vào list, `sort`, rồi lấy phần tử `[k − 1]` (chỉ số bắt đầu từ `0` nên trừ 1).

In [8]:
def kth_smallest_collect_sort(root, k):
    values = []
    stack = [root] if root else []
    while stack:                              # duyệt tùy ý (preorder bằng ngăn xếp)
        node = stack.pop()
        values.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    values.sort()                             # sắp xếp: tốn O(n log n)
    return values[k - 1]


for values, k, expected in examples:
    print(f"root={values}, k={k} -> {kth_smallest_collect_sort(build_tree(values), k)} (kỳ vọng {expected})")

root=[3, 1, 4, None, 2], k=1 -> 1 (kỳ vọng 1)
root=[5, 3, 6, 2, 4, None, None, 1], k=3 -> 3 (kỳ vọng 3)


**Độ phức tạp (Cách 1).** Gọi `n` là số nút. Duyệt `O(n)`, sắp xếp `O(n log n)`: **Time `O(n log n)`**, **Space `O(n)`** (list giá trị và ngăn xếp).

**Nút thắt.** Ta bỏ phí tính chất BST: cây **đã được sắp sẵn**, chỉ cần đọc đúng thứ tự (inorder) là có dãy tăng dần mà không phải sắp xếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Inorder của BST là dãy tăng dần.** Với mỗi nút: cây con trái chứa các giá trị nhỏ hơn, cây con phải chứa các giá trị lớn hơn. Duyệt **trái, rồi nút, rồi phải** thì mọi giá trị nhỏ hơn nút được thăm trước nút, mọi giá trị lớn hơn được thăm sau. Quy nạp theo độ cao cây cho thấy thứ tự thăm là thứ tự tăng.

Vậy giá trị nhỏ thứ `k` là **nút thứ `k` được thăm** theo inorder. Ta có các hướng:

- liệt kê toàn bộ inorder rồi lấy phần tử thứ `k` (Cách 2);
- dùng một bộ đếm `k` giảm dần và **dừng ngay khi `k = 0`** (Cách 3: stack, cách nộp bài);
- với cây thay đổi nhiều và nhiều truy vấn: lưu **kích thước cây con** ở mỗi nút để rẽ trái/phải thẳng đến đáp án trong `O(h)` (Cách 4, follow up);
- nếu muốn `O(1)` bộ nhớ phụ: **Morris traversal** (Cách 5).

In [9]:
root = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert inorder(root, []) == sorted(preorder(root, []))
print("inorder == sorted(mọi giá trị) trên BST:", inorder(root, []))

inorder == sorted(mọi giá trị) trên BST: [1, 2, 3, 4, 5, 6]


## 7. Các cách tiếp theo

### Cách 2 — Inorder đệ quy lấy cả danh sách

**Giải thích.** Viết hàm đệ quy inorder trả về list đã sắp tăng của một cây (`dfs(trái) + [gốc] + dfs(phải)`), rồi lấy phần tử `[k − 1]`. Đơn giản nhưng lấy **mọi** giá trị dù chỉ cần `k` giá trị đầu.

In [10]:
def kth_smallest_inorder_list(root, k):
    def dfs(node):
        if node is None:
            return []
        return dfs(node.left) + [node.val] + dfs(node.right)   # trái, gốc, phải
    return dfs(root)[k - 1]


for values, k, expected in examples:
    print(f"root={values}, k={k} -> {kth_smallest_inorder_list(build_tree(values), k)} (kỳ vọng {expected})")

root=[3, 1, 4, None, 2], k=1 -> 1 (kỳ vọng 1)
root=[5, 3, 6, 2, 4, None, None, 1], k=3 -> 3 (kỳ vọng 3)


**Độ phức tạp Cách 2:** **Time `O(n)`** nếu nối bằng danh sách phụ hiệu quả (ở đây phép `+` sao chép list nên tệ nhất có thể `O(n²)` cho cây lệch; nếu dùng `out.append` như `inorder` ở mục 3 thì `O(n)`), **Space `O(n)`**. Đệ quy sâu `O(h)` với `h` là độ cao cây; cây lệch có `h = n`.

### Cách 3 — Inorder bằng ngăn xếp, dừng sớm (cách của lời giải tham khảo)

**Giải thích.** Dùng stack như mục 3.4, nhưng mỗi lần **thăm** một nút thì `k -= 1`; khi `k == 0` trả giá trị ngay. Không thăm các nút còn lại.

In [11]:
def kth_smallest_stack(root, k):
    stack = []
    cur = root
    while cur or stack:
        if cur:
            stack.append(cur)                # đi sang trái
            cur = cur.left
        else:
            cur = stack.pop()                # thăm nút (nút nhỏ nhất chưa thăm)
            k -= 1
            if k == 0:
                return cur.val               # dừng sớm
            cur = cur.right                  # rồi sang cây con phải
    return None


for values, k, expected in examples:
    print(f"root={values}, k={k} -> {kth_smallest_stack(build_tree(values), k)} (kỳ vọng {expected})")

root=[3, 1, 4, None, 2], k=1 -> 1 (kỳ vọng 1)
root=[5, 3, 6, 2, 4, None, None, 1], k=3 -> 3 (kỳ vọng 3)


In [12]:
# Code nộp LeetCode cho Cách 3
from typing import Optional


class Solution:
    def kthSmallest(self, root: Optional[TreeNode], k: int) -> int:
        stack = []
        while root or stack:
            if root:
                stack.append(root)
                root = root.left
            else:
                root = stack.pop()
                k -= 1
                if k == 0:
                    return root.val
                root = root.right

**Độ phức tạp Cách 3:** **Time `O(h + k)`** (đi xuống độ sâu `h` rồi thăm `k` nút; tệ nhất `O(n)`), **Space `O(h)`** cho ngăn xếp (`h` là độ cao cây).

### Cách 4 — Đánh dấu kích thước cây con (order-statistic tree), hợp với follow up

**Giải thích.** Gọi `size(v)` là số nút trong cây con gốc `v`. Tại nút `v` với `L = size(v.left)`:

- nếu `k ≤ L`: đáp án nằm ở cây con trái, tìm nhỏ thứ `k` bên trái;
- nếu `k == L + 1`: chính `v` là đáp án;
- nếu `k > L + 1`: đáp án ở bên phải, là nhỏ thứ `k − L − 1` của cây con phải.

Sau khi đã tính `size` một lần (`O(n)`), mỗi truy vấn chỉ đi xuống một nhánh: **`O(h)`**. Khi cây bị chèn/xóa, ta cập nhật `size` dọc **đường đi** của thao tác (`O(h)`), nên truy vấn vẫn nhanh: đó là câu trả lời cho follow up.

In [13]:
def compute_sizes(root):
    """size[id(node)] = số nút của cây con gốc node. Tính bằng ngăn xếp (không đệ quy)."""
    size = {}
    order, stack = [], [root] if root else []
    while stack:                                   # preorder
        node = stack.pop()
        order.append(node)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    for node in reversed(order):                   # xử lý con trước cha
        size[id(node)] = 1 + size.get(id(node.left), 0) + size.get(id(node.right), 0)
    return size


def kth_smallest_sizes(root, k, size=None):
    if size is None:
        size = compute_sizes(root)                 # tiền xử lý O(n); bỏ qua nếu đã có sẵn
    node = root
    while node:
        left_size = size.get(id(node.left), 0)
        if k <= left_size:
            node = node.left                       # đáp án ở trái
        elif k == left_size + 1:
            return node.val                        # chính là nút này
        else:
            k -= left_size + 1                     # bỏ qua cây trái và nút này
            node = node.right
    return None


for values, k, expected in examples:
    print(f"root={values}, k={k} -> {kth_smallest_sizes(build_tree(values), k)} (kỳ vọng {expected})")

root=[3, 1, 4, None, 2], k=1 -> 1 (kỳ vọng 1)
root=[5, 3, 6, 2, 4, None, None, 1], k=3 -> 3 (kỳ vọng 3)


**Độ phức tạp Cách 4:** tiền xử lý **`O(n)`**; mỗi truy vấn **`O(h)`** (với BST cân bằng `O(log n)`), **Space `O(n)`** cho bảng `size`. Khi cây thay đổi nhiều, đây là cấu trúc "order-statistic tree".

### Cách 5 — Morris traversal (`O(1)` bộ nhớ phụ)

**Giải thích.** Không dùng ngăn xếp: tạm thời **nối** nút "tiền nhiệm inorder" (nút phải nhất của cây con trái) với nút hiện tại để biết đường quay về, và tháo dây khi đã dùng xong. Với mỗi nút `cur`: nếu không có con trái thì thăm `cur` và sang phải; nếu có con trái, tìm `pred` = nút phải nhất của cây con trái; nếu `pred.right` rỗng thì nối `pred.right = cur` rồi sang trái; nếu `pred.right == cur` (đã nối) thì tháo dây, thăm `cur`, sang phải. Cây được **khôi phục** hoàn toàn khi duyệt hết. Để mọi dây tạm được tháo, ta **không dừng sớm** khi tìm thấy đáp án mà chỉ ghi nhận nó và duyệt hết cây.

In [14]:
def kth_smallest_morris(root, k):
    cur = root
    answer = None
    while cur:
        if cur.left is None:
            k -= 1                                  # thăm cur
            if k == 0:
                answer = cur.val                    # ghi nhận nhưng KHÔNG dừng: để các dây tạm được tháo hết
            cur = cur.right
        else:
            pred = cur.left
            while pred.right and pred.right is not cur:
                pred = pred.right                   # nút phải nhất của cây con trái
            if pred.right is None:
                pred.right = cur                    # nối dây tạm để quay về cur
                cur = cur.left
            else:
                pred.right = None                   # tháo dây
                k -= 1                              # thăm cur
                if k == 0:
                    answer = cur.val
                cur = cur.right
    return answer


for values, k, expected in examples:
    tree = build_tree(values)
    got = kth_smallest_morris(tree, k)
    assert tree_to_list(tree) == values, "cây phải được khôi phục"
    print(f"root={values}, k={k} -> {got} (kỳ vọng {expected}); cây khôi phục đúng")

root=[3, 1, 4, None, 2], k=1 -> 1 (kỳ vọng 1); cây khôi phục đúng
root=[5, 3, 6, 2, 4, None, None, 1], k=3 -> 3 (kỳ vọng 3); cây khôi phục đúng


**Độ phức tạp Cách 5:** **Time `O(n)`** (mỗi cạnh được đi tối đa vài lần), **Space `O(1)`** bộ nhớ phụ (không stack, không đệ quy). Đánh đổi: tạm sửa cấu trúc cây nên phức tạp và dễ sai; vì không dừng sớm nên luôn duyệt hết `n` nút (không có lợi thế `O(h + k)` như Cách 3), đổi lại cây được khôi phục nguyên vẹn.

> Cách 3 là cách nộp bài thông dụng nhất. Cách 4 trả lời follow up. Cách 1 và 2 là chuẩn đối chiếu đơn giản.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (ngăn xếp) trên cây `[5,3,6,2,4,null,null,1]` với `k = 3`, in `stack`, nút hiện tại và `k`; rồi Cách 4 (kích thước) trên cùng cây.

In [15]:
def trace_stack(values, k):
    root = build_tree(values)
    print(f"cây={values}, k={k}")
    stack, cur = [], root
    step = 0
    while cur or stack:
        step += 1
        if cur:
            stack.append(cur)
            print(f"  bước {step}: đẩy {cur.val} vào stack -> stack={[n.val for n in stack]}, sang trái")
            cur = cur.left
        else:
            cur = stack.pop()
            k -= 1
            print(f"  bước {step}: hết bên trái, POP {cur.val} (thăm) -> còn k={k}; stack={[n.val for n in stack]}")
            if k == 0:
                print("  => k = 0, đáp án:", cur.val)
                return cur.val
            cur = cur.right
    return None


assert trace_stack([5, 3, 6, 2, 4, None, None, 1], 3) == 3

cây=[5, 3, 6, 2, 4, None, None, 1], k=3
  bước 1: đẩy 5 vào stack -> stack=[5], sang trái
  bước 2: đẩy 3 vào stack -> stack=[5, 3], sang trái
  bước 3: đẩy 2 vào stack -> stack=[5, 3, 2], sang trái
  bước 4: đẩy 1 vào stack -> stack=[5, 3, 2, 1], sang trái
  bước 5: hết bên trái, POP 1 (thăm) -> còn k=2; stack=[5, 3, 2]
  bước 6: hết bên trái, POP 2 (thăm) -> còn k=1; stack=[5, 3]
  bước 7: hết bên trái, POP 3 (thăm) -> còn k=0; stack=[5]
  => k = 0, đáp án: 3


In [16]:
def trace_sizes(values, k):
    root = build_tree(values)
    size = compute_sizes(root)
    print(f"cây={values}, k={k}")
    node = root
    while node:
        left_size = size.get(id(node.left), 0)
        print(f"  nút {node.val}: size(trái)={left_size}, k={k}", end=" -> ")
        if k <= left_size:
            print("sang trái")
            node = node.left
        elif k == left_size + 1:
            print(f"chính nút này, đáp án {node.val}")
            return node.val
        else:
            k -= left_size + 1
            print(f"bỏ trái và nút này, k mới = {k}, sang phải")
            node = node.right


assert trace_sizes([5, 3, 6, 2, 4, None, None, 1], 3) == 3
print()
assert trace_sizes([5, 3, 6, 2, 4, None, None, 1], 5) == 5

cây=[5, 3, 6, 2, 4, None, None, 1], k=3
  nút 5: size(trái)=4, k=3 -> sang trái
  nút 3: size(trái)=2, k=3 -> chính nút này, đáp án 3

cây=[5, 3, 6, 2, 4, None, None, 1], k=5
  nút 5: size(trái)=4, k=5 -> chính nút này, đáp án 5


Bảng tóm tắt (Cách 3, `k = 3`):

| bước | hành động | `stack` (từ đáy) | `k` |
|---|---|---|---|
| 1 | đẩy `5` | `[5]` | 3 |
| 2 | đẩy `3` | `[5, 3]` | 3 |
| 3 | đẩy `2` | `[5, 3, 2]` | 3 |
| 4 | đẩy `1` | `[5, 3, 2, 1]` | 3 |
| 5 | hết trái: pop `1` | `[5, 3, 2]` | 2 |
| 6 | sang phải của `1` (rỗng), pop `2` | `[5, 3]` | 1 |
| 7 | sang phải của `2` (rỗng), pop `3` | `[5]` | **0 → trả `3`** |

Chuỗi các nút được pop `1, 2, 3` chính là các giá trị nhỏ nhất theo thứ tự tăng dần.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (inorder bằng ngăn xếp, dừng sớm). Đúng class `Solution`, đúng method `kthSmallest(self, root: Optional[TreeNode], k: int) -> int`.

In [17]:
from typing import Optional


class Solution:
    def kthSmallest(self, root: Optional[TreeNode], k: int) -> int:
        stack = []
        while root or stack:
            if root:
                stack.append(root)           # đi sâu sang trái, ghi nhớ đường về
                root = root.left
            else:
                root = stack.pop()           # nút nhỏ nhất chưa được thăm
                k -= 1
                if k == 0:
                    return root.val          # đây là nút nhỏ thứ k
                root = root.right            # tiếp tục với cây con phải


for values, k, expected in examples:
    assert Solution().kthSmallest(build_tree(values), k) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Inorder là thứ tự tăng:** với BST, quy nạp theo chiều cao: cây rỗng đúng hiển nhiên; với nút `v`, inorder gồm inorder của cây trái (tăng, mọi giá trị `< v`), rồi `v`, rồi inorder của cây phải (tăng, mọi giá trị `> v`): ghép lại tăng.
- **Ngăn xếp mô phỏng đúng đệ quy inorder:** đi trái đến hết, `pop` thăm nút, rồi xử lý cây con phải của nút đó. Nút bị `pop` là nút tiếp theo theo thứ tự inorder.
- **Đếm `k` và dừng:** nút bị `pop` lần thứ `k` là nút thứ `k` trong dãy tăng, tức giá trị nhỏ thứ `k`. Vì `1 ≤ k ≤ n` nên luôn tồn tại.
- **Cách 4:** đúng nhờ `L` nút nhỏ nhất của cây tại `v` nằm hết ở cây con trái (các giá trị `< v`), nút thứ `L + 1` là `v`, phần còn lại ở cây phải với thứ hạng `k − L − 1`.
- **Cách 5:** dây tạm `pred.right = cur` cho phép quay lại `cur` sau khi hết cây con trái; lần gặp thứ hai (`pred.right is cur`) báo cây con trái đã được duyệt xong, nên thăm `cur` đúng thời điểm inorder.
- **Giá trị trùng:** nếu cây có giá trị trùng thì inorder vẫn không giảm; "nhỏ thứ `k`" theo vị trí inorder vẫn là phần tử thứ `k` của dãy đã sắp (các cách đều thống nhất).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là độ cao cây (`log n ≤ h ≤ n`).

- **Cách 1 (thu + sắp xếp):** Time `O(n log n)`, Space `O(n)`.
- **Cách 2 (inorder đệ quy lấy cả list):** Time `O(n)` (với `append`), Space `O(n)`; đệ quy sâu `O(h)` nên có thể `RecursionError` trên cây lệch.
- **Cách 3 (ngăn xếp, dừng sớm):** Time `O(h + k)`, Space `O(h)`.
- **Cách 4 (kích thước cây con):** tiền xử lý `O(n)`, truy vấn `O(h)`, Space `O(n)`.
- **Cách 5 (Morris):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất: `h ≈ log n` với cây cân bằng, `h = n` với cây lệch; vì vậy Cách 3 xấu nhất `O(n)` bộ nhớ nhưng không bị giới hạn đệ quy.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` (mỗi lần gọi dùng **cây dựng mới** vì Cách 5 tạm sửa cây). `oracle` không đọc cây: nó **sắp xếp tập giá trị đã dùng để dựng cây** rồi lấy phần tử thứ `k`. Cây ngẫu nhiên được sinh bằng `bst_insert` với các giá trị phân biệt.

In [18]:
sol = Solution()
impls = {
    "collect_sort": kth_smallest_collect_sort,
    "inorder_list": kth_smallest_inorder_list,
    "stack": kth_smallest_stack,
    "sizes": kth_smallest_sizes,
    "morris": kth_smallest_morris,
    "submit": sol.kthSmallest,
}


def random_bst(n, hi=10**4):
    values = random.sample(range(0, hi + 1), n)
    root = None
    for v in values:
        root = bst_insert(root, v)
    return root, values


def oracle(values, k):
    return sorted(values)[k - 1]


for values, k, expected in examples:
    tree_values = [v for v in values if v is not None]
    assert oracle(tree_values, k) == expected
    for name, fn in impls.items():
        assert fn(build_tree(values), k) == expected, (name, values, k)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(root_builder, values, k, expected, label, skip=()):
    for name, fn in impls.items():
        if name in skip:
            continue
        got = fn(root_builder(), k)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all(lambda: build_tree([1]), [1], 1, 1, "cây một nút")
check_all(lambda: build_tree([2, 1]), [1, 2], 1, 1, "hai nút, k = 1 (con trái)")
check_all(lambda: build_tree([2, 1]), [1, 2], 2, 2, "hai nút, k = 2 (gốc)")
check_all(lambda: build_tree([2, None, 3]), [2, 3], 2, 3, "con phải")
check_all(lambda: build_tree([3, 1, 4, None, 2]), [1, 2, 3, 4], 4, 4, "k = n: phần tử lớn nhất")
check_all(lambda: build_tree([0]), [0], 1, 0, "giá trị biên 0")
check_all(lambda: build_tree([10000]), [10000], 1, 10000, "giá trị biên 10^4")

# cây lệch phải/trái sâu 3000: đệ quy sẽ lỗi; kiểm tra các cách không dùng đệ quy
def chain_right(n):
    root = None
    for v in range(n):
        root = bst_insert(root, v)
    return root


def chain_left(n):
    root = None
    for v in range(n - 1, -1, -1):
        root = bst_insert(root, v)
    return root


for builder, label in ((lambda: chain_right(3000), "cây lệch phải 3000"), (lambda: chain_left(3000), "cây lệch trái 3000")):
    for k in (1, 1500, 3000):
        check_all(builder, list(range(3000)), k, k - 1, f"{label}, k={k}", skip=("inorder_list",))
try:
    kth_smallest_inorder_list(chain_right(3000), 5)
    print("(inorder_list không gặp lỗi đệ quy ở giới hạn hiện tại)")
except RecursionError:
    print("inorder_list: RecursionError trên cây lệch sâu 3000 (đúng như dự đoán)")

PASS cây một nút -> 1
PASS hai nút, k = 1 (con trái) -> 1
PASS hai nút, k = 2 (gốc) -> 2
PASS con phải -> 3
PASS k = n: phần tử lớn nhất -> 4
PASS giá trị biên 0 -> 0
PASS giá trị biên 10^4 -> 10000


PASS cây lệch phải 3000, k=1 -> 0


PASS cây lệch phải 3000, k=1500 -> 1499


PASS cây lệch phải 3000, k=3000 -> 2999


PASS cây lệch trái 3000, k=1 -> 0


PASS cây lệch trái 3000, k=1500 -> 1499


PASS cây lệch trái 3000, k=3000 -> 2999
inorder_list: RecursionError trên cây lệch sâu 3000 (đúng như dự đoán)


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(230)

# Tính chất 1: nhỏ thứ k tăng theo k, và nhỏ thứ 1 / nhỏ thứ n là min / max
for _ in range(100):
    n = random.randint(1, 40)
    root, values = random_bst(n)
    answers = [sol.kthSmallest(build_tree(tree_to_list(root)), k) for k in range(1, n + 1)]
    assert answers == sorted(values)
    assert answers[0] == min(values) and answers[-1] == max(values)
print("PASS tính chất: dãy đáp án theo k chính là dãy giá trị đã sắp")

# Tính chất 2: kết quả không đổi nếu dựng cùng tập giá trị theo thứ tự chèn khác (cây khác hình dạng)
for _ in range(100):
    n = random.randint(1, 30)
    values = random.sample(range(0, 1000), n)
    r1, r2 = None, None
    for v in values:
        r1 = bst_insert(r1, v)
    shuffled = values[:]
    random.shuffle(shuffled)
    for v in shuffled:
        r2 = bst_insert(r2, v)
    k = random.randint(1, n)
    assert sol.kthSmallest(r1, k) == sol.kthSmallest(r2, k) == sorted(values)[k - 1]
print("PASS tính chất: không phụ thuộc hình dạng cây")

# Phản ví dụ: dùng preorder thay cho inorder
def sai_preorder(root, k):
    return preorder(root, [])[k - 1]

print("ví dụ 2 -> sai (preorder):", sai_preorder(build_tree(examples[1][0]), 3), "| đúng:", sol.kthSmallest(build_tree(examples[1][0]), 3))
assert sai_preorder(build_tree(examples[1][0]), 3) != 3

PASS tính chất: dãy đáp án theo k chính là dãy giá trị đã sắp
PASS tính chất: không phụ thuộc hình dạng cây
ví dụ 2 -> sai (preorder): 2 | đúng: 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
for trial in range(1500):
    n = random.randint(1, 60)
    root, values = random_bst(n, hi=random.choice([100, 10**4]))
    layout = tree_to_list(root)
    k = random.randint(1, n)
    expected = oracle(values, k)
    for name, fn in impls.items():
        tree = build_tree(layout)                       # cây mới cho mỗi cách
        got = fn(tree, k)
        assert got == expected, (name, layout, k, got, expected)
        assert tree_to_list(tree) == layout, (name, "cây bị thay đổi")
for trial in range(30):                                  # cây lớn (n tới 2000), cân bằng ngẫu nhiên
    n = random.randint(1000, 2000)
    root, values = random_bst(n)
    layout = tree_to_list(root)
    k = random.randint(1, n)
    expected = oracle(values, k)
    for name, fn in impls.items():
        if name == "inorder_list" and len(layout) > 20000:
            continue
        assert fn(build_tree(layout), k) == expected, (name, n, k)
print("PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle độc lập (và cây không bị thay đổi)")

PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle độc lập (và cây không bị thay đổi)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt preorder (hoặc thứ tự khác) thay vì inorder

In [22]:
def sai_preorder_stack(root, k):
    stack = [root]
    while stack:
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)


tree_values = examples[1][0]
print("sai:", sai_preorder_stack(build_tree(tree_values), 3), "| đúng:", Solution().kthSmallest(build_tree(tree_values), 3))

sai: 2 | đúng: 3


**Vì sao sai:** preorder thăm **gốc trước**, nên giá trị thăm thứ `k` không phải nhỏ thứ `k`. Với cây ví dụ 2 preorder là `5, 3, 2, 1, 4, 6`: thứ ba là `2`. Muốn thứ tự tăng phải duyệt **inorder** (trái → gốc → phải).

### Lỗi 2: Lệch một đơn vị giữa `k` (đếm từ 1) và chỉ số list (đếm từ 0)

In [23]:
def sai_lech_mot(root, k):
    return inorder(root, [])[k]          # đúng phải là [k - 1]


print("k=1 -> sai:", sai_lech_mot(build_tree([3, 1, 4, None, 2]), 1), "| đúng:", Solution().kthSmallest(build_tree([3, 1, 4, None, 2]), 1))
try:
    sai_lech_mot(build_tree([3, 1, 4, None, 2]), 4)
except IndexError as e:
    print("k = n -> IndexError:", e)

k=1 -> sai: 2 | đúng: 1
k = n -> IndexError: list index out of range


**Vì sao sai:** đề đếm `k` từ `1`, còn list bắt đầu từ chỉ số `0`. Phần tử nhỏ thứ `k` là `list[k − 1]`. Dùng `list[k]` cho kết quả lệch một vị trí và gây `IndexError` khi `k = n`.

### Lỗi 3: Quên rẽ sang cây con phải sau khi thăm nút (hoặc quên dừng đúng chỗ)

In [24]:
def sai_quen_sang_phai(root, k):
    stack, cur = [], root
    while cur or stack:
        if cur:
            stack.append(cur)
            cur = cur.left
        else:
            cur = stack.pop()
            k -= 1
            if k == 0:
                return cur.val
            cur = None                    # sai: quên cur = cur.right


print("[5,3,6,2,4,null,null,1], k=5 -> sai:", sai_quen_sang_phai(build_tree(examples[1][0]), 5),
      "| đúng:", Solution().kthSmallest(build_tree(examples[1][0]), 5))

[5,3,6,2,4,null,null,1], k=5 -> sai: None | đúng: 5


**Vì sao sai:** sau khi thăm nút, các giá trị lớn hơn nó nằm ở **cây con phải**, nên phải `cur = cur.right`. Nếu bỏ qua, các nút nằm trong cây con phải (như `4`) không bao giờ được thăm: với `k = 5` thuật toán thăm `1, 2, 3, 5, ...` bỏ sót `4` nên trả về sai (hoặc `None`).

### Lỗi 4: Dùng biến đếm trong hàm đệ quy mà không khai báo `nonlocal`

In [25]:
def sai_khong_nonlocal(root, k):
    count = 0
    result = None

    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        count += 1                        # UnboundLocalError: count là biến cục bộ của dfs
        if count == k:
            result = node.val
        dfs(node.right)

    dfs(root)
    return result


try:
    sai_khong_nonlocal(build_tree([3, 1, 4, None, 2]), 1)
except UnboundLocalError as e:
    print("UnboundLocalError:", e)


def dung_nonlocal(root, k):
    count = 0
    result = None

    def dfs(node):
        nonlocal count, result
        if node is None or result is not None:
            return
        dfs(node.left)
        count += 1
        if count == k and result is None:
            result = node.val
        dfs(node.right)

    dfs(root)
    return result


print("dùng nonlocal ->", dung_nonlocal(build_tree([3, 1, 4, None, 2]), 1))
assert dung_nonlocal(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value
dùng nonlocal -> 1


**Vì sao sai:** gán `count += 1` trong hàm lồng nhau làm `count` thành biến **cục bộ**, chưa khởi tạo. Phải khai báo `nonlocal count, result` (hoặc dùng list một phần tử, hoặc trả giá trị). Nhớ thêm điều kiện dừng sớm khi đã tìm thấy `result`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**BST**" và "phần tử nhỏ thứ k / thứ tự sắp" → **inorder**; dừng ngay khi đếm đủ `k`.
- Cần **nhiều truy vấn** "nhỏ thứ k" trên cây thay đổi: lưu **kích thước cây con** (order-statistic tree) → `O(h)` mỗi truy vấn và mỗi cập nhật.
- Duyệt cây có thể rất sâu: ưu tiên **ngăn xếp tự quản lý** nếu lo `RecursionError` trên cây lệch (hoặc tăng `sys.setrecursionlimit`).
- Muốn `O(1)` bộ nhớ phụ: Morris traversal; đổi lại phải sửa/khôi phục cây cẩn thận.
- Muốn dựng test: viết `build_tree` (level-order ↔ cây), `tree_to_list`, và `bst_insert` để sinh cây ngẫu nhiên.
- Đối chiếu bằng **tập giá trị đã sắp** (độc lập với cấu trúc cây) là oracle rất rẻ cho mọi bài BST.

## 15. Biến thể của bài

1. **Nhỏ thứ `k` từ cuối** (lớn thứ `k`): duyệt inorder ngược (phải → gốc → trái).
2. **Kiểm tra cây có phải BST hợp lệ** (LeetCode 98): inorder phải tăng chặt.
3. **Duyệt inorder trả về cả danh sách / iterator BST** (LeetCode 173).
4. **Số nút trong khoảng `[lo, hi]`** (đếm bằng kích thước cây con hoặc cắt tỉa).
5. **Hạng (rank) của một giá trị**: có bao nhiêu giá trị nhỏ hơn `x`.

In [26]:
def kth_largest(root, k):
    """Biến thể 1: lớn thứ k (inorder ngược bằng ngăn xếp)."""
    stack, cur = [], root
    while cur or stack:
        if cur:
            stack.append(cur)
            cur = cur.right
        else:
            cur = stack.pop()
            k -= 1
            if k == 0:
                return cur.val
            cur = cur.left


def is_valid_bst(root):
    """Biến thể 2: inorder phải tăng chặt."""
    stack, cur, prev = [], root, None
    while cur or stack:
        if cur:
            stack.append(cur)
            cur = cur.left
        else:
            cur = stack.pop()
            if prev is not None and cur.val <= prev:
                return False
            prev = cur.val
            cur = cur.right
    return True


def rank_of(root, x):
    """Biến thể 5: số giá trị nhỏ hơn x trong BST."""
    count = 0
    cur = root
    while cur:
        if cur.val < x:
            count += 1 + count_nodes(cur.left)
            cur = cur.right
        else:
            cur = cur.left
    return count


def count_nodes(node):
    if node is None:
        return 0
    total, stack = 0, [node]
    while stack:
        cur = stack.pop()
        total += 1
        if cur.left:
            stack.append(cur.left)
        if cur.right:
            stack.append(cur.right)
    return total


random.seed(15)
for _ in range(200):
    n = random.randint(1, 30)
    root, values = random_bst(n, hi=200)
    k = random.randint(1, n)
    assert kth_largest(root, k) == sorted(values, reverse=True)[k - 1]
    x = random.randint(0, 200)
    assert rank_of(root, x) == sum(v < x for v in values)
    assert is_valid_bst(root)
assert is_valid_bst(build_tree([2, 1, 3])) and not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6]))
print("PASS biến thể: lớn thứ k, kiểm tra BST, rank")

PASS biến thể: lớn thứ k, kiểm tra BST, rank


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thu mọi giá trị + sắp xếp | bỏ qua tính chất BST | `O(n log n)` | `O(n)` | hiểu đề; oracle |
| 2. Inorder đệ quy lấy list | inorder = dãy tăng | `O(n)` | `O(n)` | rõ ràng, cây không quá sâu |
| 3. Inorder ngăn xếp, dừng sớm | `pop` lần thứ `k` | `O(h + k)` | `O(h)` | **cách nộp bài** |
| 4. Kích thước cây con | rẽ trái/phải theo `size(trái)` | tiền xử lý `O(n)`, mỗi truy vấn `O(h)` | `O(n)` | **follow up**: nhiều truy vấn, cây đổi |
| 5. Morris traversal | nối dây tạm, không stack | `O(n)` | `O(1)` | cần bộ nhớ phụ hằng số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `inorder(build_tree([2, 1, 3]), [])`, `preorder(build_tree([2, 1, 3]), [])`, `sorted([5, 1, 3])[1]`, `Solution().kthSmallest(build_tree([2, 1, 3]), 2)`, `Solution().kthSmallest(build_tree([2, 1, 3]), 3)`.

**Bài 2 (tính tay).** Với cây `[4, 2, 6, 1, 3, 5, 7]`: vẽ cây, viết dãy inorder và cho biết giá trị nhỏ thứ `5`. Làm lại với cây `[10, 5, 15, None, 7, 12]` và `k = 4`.

**Bài 3 (điền TODO).** Hoàn thành `kth_todo(root, k)` theo Cách 3 (ngăn xếp).

In [27]:
def kth_todo(root, k):
    stack = []
    cur = root
    while cur or stack:
        # TODO 1: nếu cur còn thì đẩy vào stack và cur = cur.left
        # TODO 2: ngược lại pop một nút, giảm k, nếu k == 0 trả cur.val
        # TODO 3: sau đó chuyển sang cur = cur.right
        pass
    return None

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `kth_sizes2(root, k)` theo Cách 4: dựng `size` rồi rẽ trái/phải theo `size(trái)`. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kth_largest2(root, k)` trả về giá trị **lớn thứ `k`**. (b) Viết `count_in_range(root, lo, hi)` đếm số nút có giá trị trong `[lo, hi]` mà **không** duyệt các nhánh chắc chắn nằm ngoài khoảng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: inorder của BST là dãy tăng; preorder thăm gốc trước. Bài 2: vẽ cây rồi duyệt trái, gốc, phải.
- **Mức 2 (vừa):** Bài 1: `[1, 2, 3]`, `[2, 1, 3]`, `3`, `2`, `3`. Bài 2: cây đầy đủ `[4,2,6,1,3,5,7]` có inorder `1..7`, nhỏ thứ 5 là `5`. Cây `[10,5,15,null,7,12]`: inorder `5, 7, 10, 12, 15`, nhỏ thứ 4 là `12`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `left_size = size.get(id(node.left), 0)`; ba trường hợp `k <= left_size`, `k == left_size + 1`, còn lại `k -= left_size + 1`. Bài 5(a): đi sang **phải** trước (đối xứng). Bài 5(b): nếu `node.val < lo` chỉ đi phải; nếu `node.val > hi` chỉ đi trái; ngược lại cộng 1 và đi cả hai.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert inorder(build_tree([2, 1, 3]), []) == [1, 2, 3]
assert preorder(build_tree([2, 1, 3]), []) == [2, 1, 3]
assert sorted([5, 1, 3])[1] == 3
assert Solution().kthSmallest(build_tree([2, 1, 3]), 2) == 2
assert Solution().kthSmallest(build_tree([2, 1, 3]), 3) == 3

# Bài 2
assert inorder(build_tree([4, 2, 6, 1, 3, 5, 7]), []) == [1, 2, 3, 4, 5, 6, 7]
assert Solution().kthSmallest(build_tree([4, 2, 6, 1, 3, 5, 7]), 5) == 5
assert inorder(build_tree([10, 5, 15, None, 7, 12]), []) == [5, 7, 10, 12, 15]
assert Solution().kthSmallest(build_tree([10, 5, 15, None, 7, 12]), 4) == 12

# Bài 3
def kth_todo_ans(root, k):
    stack = []
    cur = root
    while cur or stack:
        if cur:
            stack.append(cur)
            cur = cur.left
        else:
            cur = stack.pop()
            k -= 1
            if k == 0:
                return cur.val
            cur = cur.right
    return None

for values, k, expected in examples:
    assert kth_todo_ans(build_tree(values), k) == expected

# Bài 4
def kth_sizes2(root, k):
    size = {}
    order, stack = [], [root]
    while stack:
        node = stack.pop()
        order.append(node)
        for child in (node.left, node.right):
            if child:
                stack.append(child)
    for node in reversed(order):
        size[id(node)] = 1 + size.get(id(node.left), 0) + size.get(id(node.right), 0)
    node = root
    while node:
        left_size = size.get(id(node.left), 0)
        if k <= left_size:
            node = node.left
        elif k == left_size + 1:
            return node.val
        else:
            k -= left_size + 1
            node = node.right

for values, k, expected in examples:
    assert kth_sizes2(build_tree(values), k) == expected

# Bài 5(a), 5(b)
def kth_largest2(root, k):
    stack, cur = [], root
    while cur or stack:
        if cur:
            stack.append(cur)
            cur = cur.right
        else:
            cur = stack.pop()
            k -= 1
            if k == 0:
                return cur.val
            cur = cur.left

def count_in_range(root, lo, hi):
    total, stack = 0, [root] if root else []
    while stack:
        node = stack.pop()
        if node.val < lo:
            if node.right:
                stack.append(node.right)
        elif node.val > hi:
            if node.left:
                stack.append(node.left)
        else:
            total += 1
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
    return total

assert kth_largest2(build_tree([5, 3, 6, 2, 4, None, None, 1]), 2) == 5
random.seed(19)
for _ in range(200):
    n = random.randint(1, 30)
    root, values = random_bst(n, hi=100)
    lo, hi = sorted(random.sample(range(0, 101), 2))
    assert count_in_range(root, lo, hi) == sum(lo <= v <= hi for v in values)
    assert kth_largest2(root, 1) == max(values)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (inorder bằng ngăn xếp, dừng khi đếm đủ `k`)

```python
class Solution:
    def kthSmallest(self, root: Optional[TreeNode], k: int) -> int:
        stack = []
        while root or stack:
            if root:
                stack.append(root)
                root = root.left
            else:
                root = stack.pop()
                k -= 1
                if k == 0:
                    return root.val
                root = root.right
```

**Checklist**

- [ ] Dùng **inorder** (trái → gốc → phải), không phải preorder/postorder?
- [ ] `k` đếm từ `1`: giảm `k` khi **thăm** nút và dừng khi `k == 0` (hoặc dùng `list[k − 1]`)?
- [ ] Sau khi thăm nút, chuyển sang **cây con phải**?
- [ ] Cân nhắc **độ sâu đệ quy**: dùng ngăn xếp nếu cây có thể lệch (n tới `10⁴`)?
- [ ] Follow up: nhiều truy vấn trên cây thay đổi → lưu kích thước cây con?
- [ ] Đã thử: cây một nút, `k = 1`, `k = n`, cây lệch trái/phải?
- [ ] Đã đối chiếu với `sorted(giá trị)[k − 1]` trên cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def kthSmallest(self, root: Optional[TreeNode], k: int) -> int`?